# Inspect modeling-ready data

Loads `candidate_detail.parquet`, `features/fingerprints.parquet`, and `features/molformer_embeddings.parquet` from a pipeline + featurization run, and joins them on `candidate_id` for manual inspection.

Edit `OUT_DIR` if your `--output` lives somewhere other than `docs/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

OUT_DIR = Path('docs')
CAND_PATH  = OUT_DIR / 'candidate_detail.parquet'
FP_PATH    = OUT_DIR / 'features' / 'fingerprints.parquet'
EMB_PATH   = OUT_DIR / 'features' / 'molformer_embeddings.parquet'

for p in (CAND_PATH, FP_PATH, EMB_PATH):
    print(f'{p}: {"OK" if p.exists() else "MISSING"}')

In [ ]:
candidates = pd.read_parquet(CAND_PATH)
print('shape:', candidates.shape)
print('canonical SMILES coverage:', candidates['smiles_canonical'].notna().sum(), '/', len(candidates))
print('standardization status counts:')
print(candidates['smiles_standardization_status'].value_counts(dropna=False))
candidates[['candidate_id', 'drug_name', 'indication', 'highest_phase', 'smiles', 'smiles_canonical', 'smiles_standardization_status']].head()

In [ ]:
fingerprints = pd.read_parquet(FP_PATH) if FP_PATH.exists() else pd.DataFrame()
print('shape:', fingerprints.shape)
if len(fingerprints):
    sample = fingerprints.iloc[0]
    print('ecfp4 length:', len(sample['ecfp4']), '(expected 2048) — popcount:', int(np.array(sample['ecfp4']).sum()))
    print('maccs length:', len(sample['maccs']), '(expected 167)  — popcount:', int(np.array(sample['maccs']).sum()))
fingerprints.head()

In [ ]:
embeddings = pd.read_parquet(EMB_PATH) if EMB_PATH.exists() else pd.DataFrame()
print('shape:', embeddings.shape)
if len(embeddings):
    sample = np.array(embeddings.iloc[0]['embedding'])
    print('embedding dim:', sample.shape[0], '— mean:', float(sample.mean()), 'std:', float(sample.std()))
    print('model:', embeddings['model'].iloc[0])
embeddings.head()

In [ ]:
joined = candidates.merge(fingerprints, on='candidate_id', how='left') \
                   .merge(embeddings,   on='candidate_id', how='left')
print('joined shape:', joined.shape)
print('rows with all three:', ((joined['smiles_canonical'].notna()) & (joined['ecfp4'].notna()) & (joined['embedding'].notna())).sum())
joined[['candidate_id', 'drug_name', 'indication', 'smiles_canonical', 'ecfp4', 'embedding']].head()